# Advanced Python Properties: Deletion, State, Descriptors, and Robust Design

This notebook develops **advanced problems with complete solutions** around deleting Python properties.

The central idea is:

```python
del obj.property_name
```

does **not** remove the property from the class. Instead, Python invokes the property's **deleter** (`fdel`) on that instance.

You will work through progressively harder examples involving:

- `property(..., fdel=...)`
- `@property` / `@name.deleter`
- `delattr`
- defensive and idempotent deleters
- restoring defaults after deletion
- validation and invariants
- audit logging
- inheritance
- descriptors
- cached values
- `__slots__`
- state machines
- dependency invalidation
- testing deletion behavior
- common mistakes and debugging patterns

All cells are designed to run top-to-bottom in a fresh Python 3 notebook.

## 1. Core Mental Model

A property is a **class-level descriptor**.

If `Person.name` is a property, then:

```python
del p.name
```

roughly delegates to:

```python
Person.name.__delete__(p)
```

which calls the property's deleter function.

Deleting the instance-side backing attribute (for example `_name`) does **not** delete `Person.name` from the class.

In [1]:
class Person:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        print("GET name")
        return self._name

    @name.setter
    def name(self, value):
        print(f"SET name -> {value!r}")
        if not isinstance(value, str) or not value.strip():
            raise ValueError("name must be a non-empty string")
        self._name = value.strip()

    @name.deleter
    def name(self):
        print("DELETE name")
        del self._name


p = Person("Guido")
print("Before deletion:", p.__dict__)
print("Property value:", p.name)

del p.name

print("After deletion:", p.__dict__)
print("Property still on class:", isinstance(Person.name, property))

SET name -> 'Guido'
Before deletion: {'_name': 'Guido'}
GET name
Property value: Guido
DELETE name
After deletion: {}
Property still on class: True


### Important consequence

After deletion, the property still exists, but its getter may fail because the instance no longer has the backing state.

In [2]:
try:
    print(p.name)
except AttributeError as exc:
    print("Expected error:", exc)

GET name
Expected error: 'Person' object has no attribute '_name'


## 2. Problem 1 — Make Deletion Idempotent

### Problem

The following deleter works once, but deleting twice raises `AttributeError`.

Redesign it so repeated deletion is safe.

**Best-practice goal:** use a clear semantic policy. Here we choose: "deleting an already-missing optional value is a no-op."

In [3]:
class UserProfile:
    def __init__(self, nickname):
        self.nickname = nickname

    @property
    def nickname(self):
        return self._nickname

    @nickname.setter
    def nickname(self, value):
        self._nickname = value

    @nickname.deleter
    def nickname(self):
        # Problematic: second deletion raises AttributeError
        del self._nickname

### Solution

Use `pop` on `__dict__` with a default, or check existence explicitly.

Using `pop(..., None)` is concise when the backing attribute is stored in `__dict__`.

In [4]:
class UserProfile:
    def __init__(self, nickname):
        self.nickname = nickname

    @property
    def nickname(self):
        if "_nickname" not in self.__dict__:
            raise AttributeError("nickname has been deleted")
        return self._nickname

    @nickname.setter
    def nickname(self, value):
        if not isinstance(value, str):
            raise TypeError("nickname must be a string")
        self._nickname = value

    @nickname.deleter
    def nickname(self):
        self.__dict__.pop("_nickname", None)


u = UserProfile("py_master")
print(u.nickname)
del u.nickname
del u.nickname  # safe
print(u.__dict__)

py_master
{}


## 3. Problem 2 — Deletion Should Restore a Default

### Problem

For a configuration object, deleting `theme` should not leave the object unusable.

Instead, deletion should restore the logical default `"light"`.

This is often better than physically removing the backing attribute when the domain requires a value at all times.

In [5]:
class Settings:
    DEFAULT_THEME = "light"

    def __init__(self, theme=DEFAULT_THEME):
        self.theme = theme

    @property
    def theme(self):
        return self._theme

    @theme.setter
    def theme(self, value):
        allowed = {"light", "dark", "system"}
        if value not in allowed:
            raise ValueError(f"theme must be one of {sorted(allowed)}")
        self._theme = value

    @theme.deleter
    def theme(self):
        self._theme = self.DEFAULT_THEME


s = Settings("dark")
print("Before:", s.theme)
del s.theme
print("After:", s.theme)

Before: dark
After: light


### Design note

A deleter does **not have to call `del`**.

It can implement any deletion semantics your domain needs:

- remove a value
- reset to default
- mark data as revoked
- clear dependent caches
- write an audit entry
- refuse deletion if invariants would be violated

## 4. Problem 3 — Prevent Illegal Deletion

### Problem

Create an `Employee` class where:

- `employee_id` is readable.
- It is set during construction.
- It cannot be reassigned.
- It cannot be deleted.

Use property semantics rather than relying only on naming conventions.

In [6]:
class Employee:
    def __init__(self, employee_id):
        if not isinstance(employee_id, int) or employee_id <= 0:
            raise ValueError("employee_id must be a positive integer")
        self._employee_id = employee_id

    @property
    def employee_id(self):
        return self._employee_id

    @employee_id.deleter
    def employee_id(self):
        raise AttributeError("employee_id is immutable and cannot be deleted")


e = Employee(101)
print(e.employee_id)

try:
    e.employee_id = 999
except AttributeError as exc:
    print("Assignment blocked:", exc)

try:
    del e.employee_id
except AttributeError as exc:
    print("Deletion blocked:", exc)

101
Assignment blocked: property 'employee_id' of 'Employee' object has no setter
Deletion blocked: employee_id is immutable and cannot be deleted


## 5. Problem 4 — Delete Sensitive Data Safely

### Problem

A `Session` stores a token.

Requirements:

1. Reading the token after deletion must fail with a meaningful error.
2. Deletion should be idempotent.
3. The object should expose `is_authenticated`.
4. Reassigning a new token should restore authentication.

This models logical token revocation at the object level.

In [7]:
class Session:
    def __init__(self, token):
        self.token = token

    @property
    def token(self):
        try:
            return self._token
        except AttributeError:
            raise AttributeError("session token is not available") from None

    @token.setter
    def token(self, value):
        if not isinstance(value, str) or len(value) < 8:
            raise ValueError("token must be a string of at least 8 characters")
        self._token = value

    @token.deleter
    def token(self):
        self.__dict__.pop("_token", None)

    @property
    def is_authenticated(self):
        return "_token" in self.__dict__


session = Session("abc12345")
print(session.is_authenticated, session.token)

del session.token
print(session.is_authenticated)

try:
    print(session.token)
except AttributeError as exc:
    print(exc)

session.token = "newtoken99"
print(session.is_authenticated, session.token)

True abc12345
False
session token is not available
True newtoken99


## 6. Problem 5 — `delattr` with Dynamic Property Names

### Problem

Suppose a cleanup routine receives a list of attribute names at runtime.

Use `delattr` to clear selected properties while preserving normal deleter behavior.

In [8]:
class Contact:
    def __init__(self, email, phone):
        self.email = email
        self.phone = phone

    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, value):
        self._email = value

    @email.deleter
    def email(self):
        print("Deleting email through property deleter")
        self.__dict__.pop("_email", None)

    @property
    def phone(self):
        return self._phone

    @phone.setter
    def phone(self, value):
        self._phone = value

    @phone.deleter
    def phone(self):
        print("Deleting phone through property deleter")
        self.__dict__.pop("_phone", None)


c = Contact("a@example.com", "+359000000")
for field_name in ["email", "phone"]:
    delattr(c, field_name)

print(c.__dict__)

Deleting email through property deleter
Deleting phone through property deleter
{}


## 7. Problem 6 — Audit Every Deletion

### Problem

Build a property whose deletion:

- records an audit event,
- removes the backing value,
- remains safe on repeated deletion.

Use only the standard library.

In [9]:
from datetime import datetime, timezone


class Account:
    def __init__(self, recovery_email):
        self.audit_log = []
        self.recovery_email = recovery_email

    @property
    def recovery_email(self):
        if "_recovery_email" not in self.__dict__:
            raise AttributeError("recovery_email is not set")
        return self._recovery_email

    @recovery_email.setter
    def recovery_email(self, value):
        if "@" not in value:
            raise ValueError("invalid email")
        self._recovery_email = value

    @recovery_email.deleter
    def recovery_email(self):
        existed = "_recovery_email" in self.__dict__
        self.__dict__.pop("_recovery_email", None)
        self.audit_log.append({
            "event": "delete_recovery_email",
            "existed_before": existed,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })


account = Account("recover@example.com")
del account.recovery_email
del account.recovery_email

for event in account.audit_log:
    print(event)

{'event': 'delete_recovery_email', 'existed_before': True, 'timestamp': '2026-09-18T14:18:50.085128+00:00'}
{'event': 'delete_recovery_email', 'existed_before': False, 'timestamp': '2026-09-18T14:18:50.085223+00:00'}


## 8. Problem 7 — Invalidate Dependent Cached State

### Problem

A rectangle caches its area after first access.

Deleting `width` should:

- remove `width`,
- invalidate `_area_cache`,
- make `area` unavailable until width is assigned again.

The same principle applies to `height`.

In [10]:
class Rectangle:
    def __init__(self, width, height):
        self._area_cache = None
        self.width = width
        self.height = height

    def _invalidate_area(self):
        self._area_cache = None

    @property
    def width(self):
        if "_width" not in self.__dict__:
            raise AttributeError("width is not set")
        return self._width

    @width.setter
    def width(self, value):
        if value <= 0:
            raise ValueError("width must be positive")
        self._width = value
        self._invalidate_area()

    @width.deleter
    def width(self):
        self.__dict__.pop("_width", None)
        self._invalidate_area()

    @property
    def height(self):
        if "_height" not in self.__dict__:
            raise AttributeError("height is not set")
        return self._height

    @height.setter
    def height(self, value):
        if value <= 0:
            raise ValueError("height must be positive")
        self._height = value
        self._invalidate_area()

    @height.deleter
    def height(self):
        self.__dict__.pop("_height", None)
        self._invalidate_area()

    @property
    def area(self):
        if "_width" not in self.__dict__ or "_height" not in self.__dict__:
            raise AttributeError("area requires both width and height")
        if self._area_cache is None:
            print("Computing area...")
            self._area_cache = self._width * self._height
        return self._area_cache


r = Rectangle(4, 5)
print(r.area)
print(r.area)  # cached

del r.width

try:
    print(r.area)
except AttributeError as exc:
    print(exc)

r.width = 10
print(r.area)

Computing area...
20
20
area requires both width and height
Computing area...
50


## 9. Problem 8 — Distinguish Missing From `None`

### Problem

A common bug is using `None` to mean both:

- "the property was deleted", and
- "the property is intentionally set to `None`".

Use a sentinel object so these states are distinct.

In [11]:
_MISSING = object()


class OptionalField:
    def __init__(self, value=None):
        self._value = value

    @property
    def value(self):
        if self._value is _MISSING:
            raise AttributeError("value has been deleted")
        return self._value

    @value.setter
    def value(self, new_value):
        self._value = new_value

    @value.deleter
    def value(self):
        self._value = _MISSING


x = OptionalField(None)
print("Explicit None:", x.value)

del x.value

try:
    print(x.value)
except AttributeError as exc:
    print("Deleted state:", exc)

x.value = None
print("Restored to explicit None:", x.value)

Explicit None: None
Deleted state: value has been deleted
Restored to explicit None: None


## 10. Problem 9 — Property Deletion With Inheritance

### Problem

A base class defines a validated `name` property.

A subclass needs extra behavior when the property is deleted, but it should preserve the base deleter's logic.

Call the base property's deleter explicitly.

In [12]:
class NamedEntity:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        if "_name" not in self.__dict__:
            raise AttributeError("name is not set")
        return self._name

    @name.setter
    def name(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("name must be non-empty")
        self._name = value.strip()

    @name.deleter
    def name(self):
        self.__dict__.pop("_name", None)


class TrackedEntity(NamedEntity):
    def __init__(self, name):
        self.events = []
        super().__init__(name)

    @NamedEntity.name.deleter
    def name(self):
        self.events.append("name_deleted")
        # Call the base property's fdel directly.
        NamedEntity.name.fdel(self)


obj = TrackedEntity("alpha")
del obj.name
print(obj.__dict__)
print(obj.events)

{'events': ['name_deleted']}
['name_deleted']


### Why this works

`NamedEntity.name` is a property object.

Its `fdel` attribute references the base deleter function, so:

```python
NamedEntity.name.fdel(self)
```

reuses the original deletion behavior.

This is useful when extending property semantics in subclasses.

## 11. Problem 10 — Deletion in a State Machine

### Problem

Create a `Document` where deleting `approval_code` is allowed only while the document is in `"draft"` state.

Once the document is `"approved"`, deletion must be rejected.

In [13]:
class Document:
    def __init__(self, approval_code=None):
        self.status = "draft"
        if approval_code is not None:
            self.approval_code = approval_code

    @property
    def approval_code(self):
        if "_approval_code" not in self.__dict__:
            raise AttributeError("approval_code is not set")
        return self._approval_code

    @approval_code.setter
    def approval_code(self, value):
        if self.status != "draft":
            raise RuntimeError("approval_code can only be changed in draft state")
        if not isinstance(value, str) or not value:
            raise ValueError("approval_code must be non-empty")
        self._approval_code = value

    @approval_code.deleter
    def approval_code(self):
        if self.status != "draft":
            raise RuntimeError("approval_code cannot be deleted after approval")
        self.__dict__.pop("_approval_code", None)

    def approve(self):
        if "_approval_code" not in self.__dict__:
            raise RuntimeError("cannot approve without approval_code")
        self.status = "approved"


doc = Document("A-123")
del doc.approval_code
doc.approval_code = "A-456"
doc.approve()

try:
    del doc.approval_code
except RuntimeError as exc:
    print(exc)

approval_code cannot be deleted after approval


## 12. Problem 11 — Property Deletion With `__slots__`

### Problem

Classes using `__slots__` may not have a `__dict__`.

Therefore patterns like:

```python
self.__dict__.pop("_value", None)
```

may fail.

Implement a property deleter for a slotted class.

In [14]:
class SlottedToken:
    __slots__ = ("_token",)

    def __init__(self, token):
        self.token = token

    @property
    def token(self):
        try:
            return self._token
        except AttributeError:
            raise AttributeError("token has been deleted") from None

    @token.setter
    def token(self, value):
        if not isinstance(value, str) or not value:
            raise ValueError("token must be a non-empty string")
        self._token = value

    @token.deleter
    def token(self):
        if hasattr(self, "_token"):
            del self._token


st = SlottedToken("xyz")
print(st.token)
del st.token
del st.token  # idempotent

try:
    print(st.token)
except AttributeError as exc:
    print(exc)

xyz
token has been deleted


## 13. Problem 12 — Deleting a Property That Manages Multiple Attributes

### Problem

A `FullName` property is backed by `_first_name` and `_last_name`.

Deleting `full_name` should remove both pieces of state atomically from the instance.

In [15]:
class PersonName:
    def __init__(self, first_name, last_name):
        self.full_name = (first_name, last_name)

    @property
    def full_name(self):
        if not hasattr(self, "_first_name") or not hasattr(self, "_last_name"):
            raise AttributeError("full_name has been deleted")
        return f"{self._first_name} {self._last_name}"

    @full_name.setter
    def full_name(self, value):
        try:
            first, last = value
        except (TypeError, ValueError):
            raise ValueError("full_name must be a 2-item iterable") from None

        first = str(first).strip()
        last = str(last).strip()

        if not first or not last:
            raise ValueError("both first and last name are required")

        self._first_name = first
        self._last_name = last

    @full_name.deleter
    def full_name(self):
        for attribute in ("_first_name", "_last_name"):
            if hasattr(self, attribute):
                delattr(self, attribute)


pn = PersonName("Ada", "Lovelace")
print(pn.full_name)
del pn.full_name
print(vars(pn))

Ada Lovelace
{}


## 14. Problem 13 — Soft Delete Instead of Physical Delete

### Problem

Sometimes deletion must preserve historical information.

Implement a property where `del obj.email` makes the email unavailable to normal callers but archives the previous value internally.

In [16]:
class Customer:
    def __init__(self, email):
        self._archived_emails = []
        self.email = email

    @property
    def email(self):
        if "_email" not in self.__dict__:
            raise AttributeError("email is inactive")
        return self._email

    @email.setter
    def email(self, value):
        if not isinstance(value, str) or "@" not in value:
            raise ValueError("invalid email")
        self._email = value

    @email.deleter
    def email(self):
        if "_email" in self.__dict__:
            self._archived_emails.append(self._email)
            del self._email

    @property
    def archived_emails(self):
        return tuple(self._archived_emails)


customer = Customer("old@example.com")
del customer.email
print(customer.archived_emails)

customer.email = "new@example.com"
print(customer.email)
print(customer.archived_emails)

('old@example.com',)
new@example.com
('old@example.com',)


## 15. Problem 14 — Transaction-Like Deletion With Invariants

### Problem

An order has:

- `billing_address`
- `shipping_address`

If `use_billing_for_shipping` is `True`, deleting `billing_address` would break an invariant because shipping depends on billing.

Reject deletion in that case.

In [17]:
class Order:
    def __init__(self, billing_address, shipping_address=None, use_billing_for_shipping=False):
        self.use_billing_for_shipping = use_billing_for_shipping
        self.billing_address = billing_address
        if shipping_address is not None:
            self.shipping_address = shipping_address

    @property
    def billing_address(self):
        if "_billing_address" not in self.__dict__:
            raise AttributeError("billing_address is not set")
        return self._billing_address

    @billing_address.setter
    def billing_address(self, value):
        if not value:
            raise ValueError("billing_address cannot be empty")
        self._billing_address = value

    @billing_address.deleter
    def billing_address(self):
        if self.use_billing_for_shipping:
            raise RuntimeError(
                "cannot delete billing_address while shipping depends on it"
            )
        self.__dict__.pop("_billing_address", None)

    @property
    def shipping_address(self):
        if self.use_billing_for_shipping:
            return self.billing_address
        if "_shipping_address" not in self.__dict__:
            raise AttributeError("shipping_address is not set")
        return self._shipping_address

    @shipping_address.setter
    def shipping_address(self, value):
        if not value:
            raise ValueError("shipping_address cannot be empty")
        self._shipping_address = value


order = Order("Billing St", use_billing_for_shipping=True)
print(order.shipping_address)

try:
    del order.billing_address
except RuntimeError as exc:
    print(exc)

Billing St
cannot delete billing_address while shipping depends on it


## 16. Problem 15 — A Reusable Descriptor With `__delete__`

Properties are themselves descriptors, but sometimes a custom descriptor is better when the same validation/deletion behavior must be reused across many fields.

### Problem

Build a descriptor that:

- validates values,
- stores data under a private name,
- supports deletion,
- raises a useful error when read after deletion.

In [18]:
class ValidatedField:
    def __init__(self, validator):
        self.validator = validator
        self.public_name = None
        self.private_name = None

    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if not hasattr(instance, self.private_name):
            raise AttributeError(f"{self.public_name} has been deleted")
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        if not self.validator(value):
            raise ValueError(f"invalid value for {self.public_name}: {value!r}")
        setattr(instance, self.private_name, value)

    def __delete__(self, instance):
        if hasattr(instance, self.private_name):
            delattr(instance, self.private_name)


def positive_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and value > 0


class Product:
    price = ValidatedField(positive_number)
    weight = ValidatedField(positive_number)

    def __init__(self, price, weight):
        self.price = price
        self.weight = weight


product = Product(price=19.99, weight=2.5)
print(product.price, product.weight)

del product.price
print(vars(product))

try:
    print(product.price)
except AttributeError as exc:
    print(exc)

19.99 2.5
{'_weight': 2.5}
price has been deleted


## 17. Problem 16 — Descriptor Deletion With an Audit Hook

### Problem

Extend the descriptor idea so every deletion can notify the owning instance.

This avoids duplicating audit logic in many property deleters.

In [19]:
class AuditedField:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if not hasattr(instance, self.private_name):
            raise AttributeError(f"{self.public_name} is not set")
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        setattr(instance, self.private_name, value)

    def __delete__(self, instance):
        existed = hasattr(instance, self.private_name)
        old_value = getattr(instance, self.private_name, None)

        if existed:
            delattr(instance, self.private_name)

        hook = getattr(instance, "_on_field_deleted", None)
        if hook is not None:
            hook(self.public_name, old_value, existed)


class Record:
    secret = AuditedField()
    label = AuditedField()

    def __init__(self, secret, label):
        self.events = []
        self.secret = secret
        self.label = label

    def _on_field_deleted(self, field, old_value, existed):
        self.events.append({
            "field": field,
            "old_value": old_value,
            "existed": existed,
        })


record = Record("s3cr3t", "demo")
del record.secret
del record.secret
print(record.events)

[{'field': 'secret', 'old_value': 's3cr3t', 'existed': True}, {'field': 'secret', 'old_value': None, 'existed': False}]


## 18. Problem 17 — Lazy Resource Lifecycle

### Problem

A property lazily creates a resource only when accessed.

Deleting the property should close the resource and remove it from the instance.

This pattern is useful for file handles, connections, sockets, or other lifecycle-managed resources.

In [20]:
class FakeResource:
    def __init__(self):
        self.closed = False
        print("Resource created")

    def close(self):
        if not self.closed:
            self.closed = True
            print("Resource closed")


class ResourceOwner:
    @property
    def resource(self):
        if "_resource" not in self.__dict__:
            self._resource = FakeResource()
        return self._resource

    @resource.deleter
    def resource(self):
        resource = self.__dict__.pop("_resource", None)
        if resource is not None:
            resource.close()


owner = ResourceOwner()
print("Initially:", owner.__dict__)

resource = owner.resource
print("After access:", owner.__dict__)

del owner.resource
print("After delete:", owner.__dict__)
print("Original resource closed:", resource.closed)

Initially: {}
Resource created
After access: {'_resource': <__main__.FakeResource object at 0x00000285017EC1A0>}
Resource closed
After delete: {}
Original resource closed: True


### Best-practice warning

For real external resources, context managers (`with ...:`) are often clearer and safer than depending on property deletion.

A property deleter can still be useful when the object explicitly owns a long-lived optional resource.

## 19. Problem 18 — Deleting a Cached Computation

### Problem

Make `report` a lazily computed property.

`del obj.report` should invalidate only the cached report, not the raw source data.

The next read should recompute it.

In [21]:
class Analyzer:
    def __init__(self, values):
        self.values = list(values)

    @property
    def report(self):
        if "_report_cache" not in self.__dict__:
            print("Computing report...")
            values = self.values
            self._report_cache = {
                "count": len(values),
                "sum": sum(values),
                "mean": sum(values) / len(values) if values else None,
            }
        return self._report_cache

    @report.deleter
    def report(self):
        self.__dict__.pop("_report_cache", None)


a = Analyzer([10, 20, 30])
print(a.report)
print(a.report)

del a.report

a.values.append(40)
print(a.report)

Computing report...
{'count': 3, 'sum': 60, 'mean': 20.0}
{'count': 3, 'sum': 60, 'mean': 20.0}
Computing report...
{'count': 4, 'sum': 100, 'mean': 25.0}


## 20. Problem 19 — Why This Deleter Is Buggy

### Problem

Explain what is wrong with this class:

```python
class Broken:
    @property
    def value(self):
        return self.value

    @value.setter
    def value(self, x):
        self.value = x

    @value.deleter
    def value(self):
        del self.value
```

Then fix it.

### Solution

Each accessor recursively refers to the property itself:

- getter calls getter again,
- setter calls setter again,
- deleter calls deleter again.

This leads to `RecursionError`.

Use a distinct backing attribute such as `_value`.

In [22]:
class Fixed:
    def __init__(self, value):
        self.value = value

    @property
    def value(self):
        if "_value" not in self.__dict__:
            raise AttributeError("value has been deleted")
        return self._value

    @value.setter
    def value(self, x):
        self._value = x

    @value.deleter
    def value(self):
        self.__dict__.pop("_value", None)


f = Fixed(42)
print(f.value)
del f.value
print(f.__dict__)

42
{}


## 21. Problem 20 — Property Factory With Getter, Setter, and Deleter

### Problem

Create a helper function that returns a property object for a positive numeric field.

Use the classic `property(fget=..., fset=..., fdel=..., doc=...)` form rather than decorators.

In [23]:
def positive_property(storage_name, doc):
    def getter(self):
        if not hasattr(self, storage_name):
            raise AttributeError(f"{storage_name} is not set")
        return getattr(self, storage_name)

    def setter(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool) or value <= 0:
            raise ValueError("value must be a positive number")
        setattr(self, storage_name, value)

    def deleter(self):
        if hasattr(self, storage_name):
            delattr(self, storage_name)

    return property(
        fget=getter,
        fset=setter,
        fdel=deleter,
        doc=doc,
    )


class Box:
    width = positive_property("_width", "Box width")
    height = positive_property("_height", "Box height")

    def __init__(self, width, height):
        self.width = width
        self.height = height


b = Box(3, 4)
print(b.width, b.height)
del b.width
print(vars(b))
print(Box.width.__doc__)

3 4
{'_height': 4}
Box width


## 22. Problem 21 — Introspect Property Accessors

### Problem

Inspect a property object to determine whether it has getter, setter, and deleter functions.

In [24]:
class Demo:
    @property
    def x(self):
        return 1

    @x.setter
    def x(self, value):
        pass

    @x.deleter
    def x(self):
        pass


prop = Demo.x

print("is property:", isinstance(prop, property))
print("getter:", prop.fget)
print("setter:", prop.fset)
print("deleter:", prop.fdel)
print("doc:", prop.__doc__)

is property: True
getter: <function Demo.x at 0x00000285716ACB80>
setter: <function Demo.x at 0x00000285017DD580>
deleter: <function Demo.x at 0x00000285017DD120>
doc: None


## 23. Problem 22 — Read-Only Property With a Deleter

A property can have a getter and deleter without having a setter.

### Problem

Create a generated identifier that cannot be assigned directly, but can be deleted so that a new identifier is generated on the next read.

In [25]:
from uuid import uuid4


class Entity:
    @property
    def identifier(self):
        if "_identifier" not in self.__dict__:
            self._identifier = str(uuid4())
        return self._identifier

    @identifier.deleter
    def identifier(self):
        self.__dict__.pop("_identifier", None)


entity = Entity()

first = entity.identifier
print("first:", first)

try:
    entity.identifier = "manual-id"
except AttributeError as exc:
    print("Assignment blocked:", exc)

del entity.identifier

second = entity.identifier
print("second:", second)
print("changed:", first != second)

first: fdbc9f29-aa46-40fe-a873-a948944f4bef
Assignment blocked: property 'identifier' of 'Entity' object has no setter
second: eb13e0a6-4218-42bf-91ba-352eed7db64b
changed: True


## 24. Problem 23 — Deletion Policy Based on Object Ownership

### Problem

An API key can only be deleted by its owner.

Model this by making deletion require a temporary authorization context stored on the object.

This is a teaching example for policy enforcement; real authorization systems should not rely only on mutable in-memory flags.

In [26]:
from contextlib import contextmanager


class ApiCredential:
    def __init__(self, owner, api_key):
        self.owner = owner
        self.api_key = api_key
        self._acting_user = None

    @contextmanager
    def acting_as(self, username):
        previous = self._acting_user
        self._acting_user = username
        try:
            yield self
        finally:
            self._acting_user = previous

    @property
    def api_key(self):
        if "_api_key" not in self.__dict__:
            raise AttributeError("api_key has been deleted")
        return self._api_key

    @api_key.setter
    def api_key(self, value):
        if not value:
            raise ValueError("api_key cannot be empty")
        self._api_key = value

    @api_key.deleter
    def api_key(self):
        if self._acting_user != self.owner:
            raise PermissionError("only the owner may delete the api_key")
        self.__dict__.pop("_api_key", None)


cred = ApiCredential("alice", "key-123")

try:
    del cred.api_key
except PermissionError as exc:
    print(exc)

with cred.acting_as("alice"):
    del cred.api_key

print(cred.__dict__)

only the owner may delete the api_key
{'owner': 'alice', '_acting_user': None}


## 25. Problem 24 — Deletion and `hasattr`

### Problem

Understand how `hasattr(obj, "prop")` behaves when the property's getter raises `AttributeError`.

`hasattr` calls `getattr` internally and returns `False` when an `AttributeError` occurs.

In [27]:
class HasAttrDemo:
    def __init__(self):
        self.value = 10

    @property
    def value(self):
        if "_value" not in self.__dict__:
            raise AttributeError("value missing")
        return self._value

    @value.setter
    def value(self, x):
        self._value = x

    @value.deleter
    def value(self):
        self.__dict__.pop("_value", None)


h = HasAttrDemo()
print("Before:", hasattr(h, "value"))
del h.value
print("After:", hasattr(h, "value"))

Before: True
After: False


### Subtle point

If a property getter raises a different exception such as `ValueError`, `hasattr` does **not** suppress it.

Use `AttributeError` to represent a missing attribute/property state.

## 26. Problem 25 — Build a Resettable Secret With Versioning

### Problem

Create a `VersionedSecret` class.

Requirements:

- assigning a secret increments `version`,
- deleting it increments `version`,
- deleting an already-missing secret does not increment,
- reading a missing secret raises `AttributeError`,
- `has_secret` reports the state.

In [28]:
class VersionedSecret:
    def __init__(self, secret=None):
        self.version = 0
        if secret is not None:
            self.secret = secret

    @property
    def secret(self):
        if "_secret" not in self.__dict__:
            raise AttributeError("secret is not set")
        return self._secret

    @secret.setter
    def secret(self, value):
        if not isinstance(value, str) or not value:
            raise ValueError("secret must be a non-empty string")
        self._secret = value
        self.version += 1

    @secret.deleter
    def secret(self):
        if "_secret" in self.__dict__:
            del self._secret
            self.version += 1

    @property
    def has_secret(self):
        return "_secret" in self.__dict__


vs = VersionedSecret("alpha")
print(vs.version, vs.has_secret)

del vs.secret
print(vs.version, vs.has_secret)

del vs.secret
print(vs.version, vs.has_secret)

vs.secret = "beta"
print(vs.version, vs.secret)

1 True
2 False
2 False
3 beta


# Testing Property Deletion

A good property deleter should be tested for:

1. state before deletion,
2. state after deletion,
3. behavior of the getter afterward,
4. repeated deletion if idempotence is intended,
5. reassignment after deletion if supported,
6. side effects such as logs or cache invalidation,
7. invariants and forbidden deletion paths.

## 27. Problem 26 — Write Unit Tests With `unittest`

### Problem

Test the `VersionedSecret` behavior with the standard library's `unittest`.

In [29]:
import unittest


class TestVersionedSecret(unittest.TestCase):
    def test_delete_existing_secret(self):
        obj = VersionedSecret("alpha")
        original_version = obj.version

        del obj.secret

        self.assertFalse(obj.has_secret)
        self.assertEqual(obj.version, original_version + 1)

        with self.assertRaises(AttributeError):
            _ = obj.secret

    def test_delete_missing_secret_is_idempotent(self):
        obj = VersionedSecret("alpha")
        del obj.secret
        version_after_first_delete = obj.version

        del obj.secret

        self.assertEqual(obj.version, version_after_first_delete)

    def test_reassign_after_delete(self):
        obj = VersionedSecret("alpha")
        del obj.secret

        obj.secret = "beta"

        self.assertTrue(obj.has_secret)
        self.assertEqual(obj.secret, "beta")


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestVersionedSecret)
result = unittest.TextTestRunner(verbosity=2).run(suite)
print("Successful:", result.wasSuccessful())

test_delete_existing_secret (__main__.TestVersionedSecret.test_delete_existing_secret) ... ok
test_delete_missing_secret_is_idempotent (__main__.TestVersionedSecret.test_delete_missing_secret_is_idempotent) ... ok
test_reassign_after_delete (__main__.TestVersionedSecret.test_reassign_after_delete) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.009s

OK


Successful: True


## 28. Problem 27 — Compare `del obj.prop` and `delattr(obj, "prop")`

### Problem

Prove that both forms trigger the same property deleter.

In [30]:
class Counter:
    def __init__(self):
        self.delete_calls = 0
        self.value = 1

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, x):
        self._value = x

    @value.deleter
    def value(self):
        self.delete_calls += 1
        self.__dict__.pop("_value", None)


c1 = Counter()
del c1.value
print("del calls:", c1.delete_calls)

c2 = Counter()
delattr(c2, "value")
print("delattr calls:", c2.delete_calls)

del calls: 1
delattr calls: 1


## 29. Problem 28 — Deleter Calls Another Property Safely

### Problem

Deleting `raw_data` must also invalidate a derived `summary`.

Do not duplicate the summary invalidation logic.

In [31]:
class Dataset:
    def __init__(self, raw_data):
        self.raw_data = raw_data

    @property
    def raw_data(self):
        if "_raw_data" not in self.__dict__:
            raise AttributeError("raw_data is not set")
        return self._raw_data

    @raw_data.setter
    def raw_data(self, value):
        self._raw_data = list(value)
        # New raw data makes any old summary stale.
        del self.summary

    @raw_data.deleter
    def raw_data(self):
        self.__dict__.pop("_raw_data", None)
        del self.summary

    @property
    def summary(self):
        if "_summary" not in self.__dict__:
            if "_raw_data" not in self.__dict__:
                raise AttributeError("summary requires raw_data")
            values = self._raw_data
            self._summary = {
                "count": len(values),
                "min": min(values) if values else None,
                "max": max(values) if values else None,
            }
        return self._summary

    @summary.deleter
    def summary(self):
        self.__dict__.pop("_summary", None)


data = Dataset([3, 1, 9])
print(data.summary)
print(data.__dict__)

del data.raw_data
print(data.__dict__)

{'count': 3, 'min': 1, 'max': 9}
{'_raw_data': [3, 1, 9], '_summary': {'count': 3, 'min': 1, 'max': 9}}
{}


## 30. Problem 29 — Multi-Level Cache Invalidation

### Problem

A `Circle` has:

- `radius`
- cached `area`
- cached `circumference`

Changing or deleting radius must invalidate both caches.

Centralize invalidation to avoid bugs.

In [32]:
import math


class Circle:
    def __init__(self, radius):
        self.radius = radius

    def _invalidate_geometry(self):
        self.__dict__.pop("_area_cache", None)
        self.__dict__.pop("_circumference_cache", None)

    @property
    def radius(self):
        if "_radius" not in self.__dict__:
            raise AttributeError("radius is not set")
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("radius must be positive")
        self._radius = value
        self._invalidate_geometry()

    @radius.deleter
    def radius(self):
        self.__dict__.pop("_radius", None)
        self._invalidate_geometry()

    @property
    def area(self):
        if "_area_cache" not in self.__dict__:
            self._area_cache = math.pi * self.radius ** 2
        return self._area_cache

    @property
    def circumference(self):
        if "_circumference_cache" not in self.__dict__:
            self._circumference_cache = 2 * math.pi * self.radius
        return self._circumference_cache


circle = Circle(2)
print(circle.area)
print(circle.circumference)
print(circle.__dict__)

del circle.radius
print(circle.__dict__)

12.566370614359172
12.566370614359172
{'_radius': 2, '_area_cache': 12.566370614359172, '_circumference_cache': 12.566370614359172}
{}


## 31. Problem 30 — Avoid Leaking Internal Representation

### Problem

A getter should not expose internal storage assumptions to callers.

Return a domain-specific `AttributeError` rather than the default message about `_email`.

In [33]:
class BetterErrors:
    def __init__(self, email):
        self.email = email

    @property
    def email(self):
        try:
            return self._email
        except AttributeError:
            raise AttributeError("email is not currently configured") from None

    @email.setter
    def email(self, value):
        self._email = value

    @email.deleter
    def email(self):
        if hasattr(self, "_email"):
            del self._email


be = BetterErrors("person@example.com")
del be.email

try:
    print(be.email)
except AttributeError as exc:
    print(exc)

email is not currently configured


# Challenge Set

Try each problem before opening its solution cell.

## Challenge 1 — Temporary Override

Implement `FeatureFlag` so that:

- `enabled` normally returns a default value,
- assigning `enabled` creates an instance override,
- deleting `enabled` removes only the override,
- after deletion, reads fall back to the default.

### Challenge 1 Solution

In [34]:
class FeatureFlag:
    def __init__(self, default=False):
        self.default = bool(default)

    @property
    def enabled(self):
        return self.__dict__.get("_enabled_override", self.default)

    @enabled.setter
    def enabled(self, value):
        if not isinstance(value, bool):
            raise TypeError("enabled must be bool")
        self._enabled_override = value

    @enabled.deleter
    def enabled(self):
        self.__dict__.pop("_enabled_override", None)


flag = FeatureFlag(default=False)
print(flag.enabled)

flag.enabled = True
print(flag.enabled)

del flag.enabled
print(flag.enabled)

False
True
False


## Challenge 2 — Prevent Deleting the Last Contact Method

Build a profile with optional `email` and `phone`.

Deletion is allowed only if at least one contact method remains.

### Challenge 2 Solution

In [35]:
class ContactProfile:
    def __init__(self, email=None, phone=None):
        if email is None and phone is None:
            raise ValueError("at least one contact method is required")
        if email is not None:
            self.email = email
        if phone is not None:
            self.phone = phone

    def _contact_count(self):
        return int("_email" in self.__dict__) + int("_phone" in self.__dict__)

    @property
    def email(self):
        if "_email" not in self.__dict__:
            raise AttributeError("email is not set")
        return self._email

    @email.setter
    def email(self, value):
        if "@" not in value:
            raise ValueError("invalid email")
        self._email = value

    @email.deleter
    def email(self):
        if "_email" not in self.__dict__:
            return
        if self._contact_count() == 1:
            raise RuntimeError("cannot delete the last contact method")
        del self._email

    @property
    def phone(self):
        if "_phone" not in self.__dict__:
            raise AttributeError("phone is not set")
        return self._phone

    @phone.setter
    def phone(self, value):
        if not value:
            raise ValueError("invalid phone")
        self._phone = value

    @phone.deleter
    def phone(self):
        if "_phone" not in self.__dict__:
            return
        if self._contact_count() == 1:
            raise RuntimeError("cannot delete the last contact method")
        del self._phone


profile = ContactProfile(email="a@example.com", phone="+123")
del profile.email
print(profile.__dict__)

try:
    del profile.phone
except RuntimeError as exc:
    print(exc)

{'_phone': '+123'}
cannot delete the last contact method


## Challenge 3 — Cascading Deletion

A `User` has a `username` and a cached `profile_url`.

Deleting `username` must also delete the URL cache.

Reassigning `username` should allow `profile_url` to work again.

### Challenge 3 Solution

In [36]:
from urllib.parse import quote


class User:
    def __init__(self, username):
        self.username = username

    @property
    def username(self):
        if "_username" not in self.__dict__:
            raise AttributeError("username is not set")
        return self._username

    @username.setter
    def username(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("username cannot be empty")
        self._username = value
        self.__dict__.pop("_profile_url", None)

    @username.deleter
    def username(self):
        self.__dict__.pop("_username", None)
        self.__dict__.pop("_profile_url", None)

    @property
    def profile_url(self):
        if "_profile_url" not in self.__dict__:
            self._profile_url = f"https://example.com/u/{quote(self.username)}"
        return self._profile_url


user = User("Ada Lovelace")
print(user.profile_url)

del user.username
print(user.__dict__)

user.username = "Grace Hopper"
print(user.profile_url)

https://example.com/u/Ada%20Lovelace
{}
https://example.com/u/Grace%20Hopper


## Challenge 4 — Reusable Resettable Property Factory

Create a factory `resettable_property(...)` that:

- validates values,
- stores them under a chosen private name,
- deletes by restoring a default,
- supports a custom docstring.

### Challenge 4 Solution

In [37]:
def resettable_property(storage_name, *, default, validator=lambda x: True, doc=None):
    def getter(self):
        return getattr(self, storage_name, default)

    def setter(self, value):
        if not validator(value):
            raise ValueError(f"invalid value: {value!r}")
        setattr(self, storage_name, value)

    def deleter(self):
        setattr(self, storage_name, default)

    return property(getter, setter, deleter, doc)


class Preferences:
    language = resettable_property(
        "_language",
        default="en",
        validator=lambda x: isinstance(x, str) and len(x) == 2,
        doc="Two-letter language code.",
    )

    page_size = resettable_property(
        "_page_size",
        default=20,
        validator=lambda x: isinstance(x, int) and not isinstance(x, bool) and 1 <= x <= 100,
        doc="Number of rows per page.",
    )


prefs = Preferences()
print(prefs.language, prefs.page_size)

prefs.language = "bg"
prefs.page_size = 50
print(prefs.language, prefs.page_size)

del prefs.language
del prefs.page_size
print(prefs.language, prefs.page_size)

en 20
bg 50
en 20


## Challenge 5 — Property Deletion + Event Listeners

Implement a model where external listeners can subscribe to property deletion events.

Do not hard-code print statements inside the property.

### Challenge 5 Solution

In [38]:
class ObservableProfile:
    def __init__(self, bio):
        self._listeners = []
        self.bio = bio

    def add_listener(self, callback):
        if not callable(callback):
            raise TypeError("listener must be callable")
        self._listeners.append(callback)

    def _emit(self, event, **payload):
        for callback in tuple(self._listeners):
            callback(event, payload)

    @property
    def bio(self):
        if "_bio" not in self.__dict__:
            raise AttributeError("bio has been deleted")
        return self._bio

    @bio.setter
    def bio(self, value):
        self._bio = str(value)

    @bio.deleter
    def bio(self):
        if "_bio" not in self.__dict__:
            return

        old_value = self._bio
        del self._bio
        self._emit("bio_deleted", old_value=old_value)


events = []

profile = ObservableProfile("Python developer")
profile.add_listener(lambda event, payload: events.append((event, payload)))

del profile.bio
print(events)

[('bio_deleted', {'old_value': 'Python developer'})]


# Common Mistakes

## Mistake 1: Confusing property deletion with class modification

```python
del instance.name
```

does not do this:

```python
del Class.name
```

The first invokes instance-level descriptor deletion.  
The second actually removes the attribute/property from the class namespace.

In [39]:
class Example:
    def __init__(self):
        self._x = 1

    @property
    def x(self):
        return self._x

    @x.deleter
    def x(self):
        del self._x


obj = Example()

print("Before instance deletion:", "x" in Example.__dict__)
del obj.x
print("After instance deletion:", "x" in Example.__dict__)

Before instance deletion: True
After instance deletion: True


## Mistake 2: Using a property without a deleter

If a property has no `fdel`, deletion raises `AttributeError`.

In [40]:
class NoDeleter:
    @property
    def value(self):
        return 123


nd = NoDeleter()

try:
    del nd.value
except AttributeError as exc:
    print(exc)

property 'value' of 'NoDeleter' object has no deleter


## Mistake 3: Accidentally bypassing the property

This deletes the backing attribute directly:

```python
del obj._name
```

It bypasses all logic in:

```python
@name.deleter
```

Callers should normally delete the public property:

```python
del obj.name
```

## Mistake 4: Swallowing every exception

Avoid this:

```python
try:
    del self._value
except Exception:
    pass
```

It can hide programming errors.

Prefer a specific check:

```python
if hasattr(self, "_value"):
    del self._value
```

or, for normal `__dict__`-backed objects:

```python
self.__dict__.pop("_value", None)
```

# Best-Practice Checklist

When designing a deletable property:

1. **Define deletion semantics explicitly.** Does deletion mean "missing", "default", "revoked", "archived", or "closed"?
2. **Use a distinct backing name.** Usually `_name`, never the property name itself.
3. **Preserve invariants.** Reject deletion when it would leave the object invalid.
4. **Decide whether deletion is idempotent.** For optional state, idempotent deletion is often convenient.
5. **Use `AttributeError` for missing attribute state.** This works naturally with `hasattr`.
6. **Invalidate dependent caches.** Deleting source data should clear derived state.
7. **Be careful with `__slots__`.** A slotted object may not have `__dict__`.
8. **Avoid unnecessary side effects.** Deleters should remain understandable and predictable.
9. **Test repeated delete / reassign cycles.**
10. **Prefer context managers for short-lived external resources.**
11. **Use descriptors when the same field behavior is repeated across many classes.**
12. **Do not confuse deleting an instance property's value with deleting the property from the class.**

# Final Integrated Exercise

## Build a Production-Style `ApiClientConfig`

Requirements:

- `base_url` is required and cannot be deleted.
- `api_token` is optional and may be deleted idempotently.
- Reading a deleted token raises `AttributeError`.
- `timeout` must be positive and deleting it restores the default.
- `headers` is lazily computed and cached.
- Changing/deleting `api_token` invalidates `headers`.
- All token deletion events are audited.
- A `reset_optional_state()` method should use public deletion behavior rather than directly manipulating private names.

## Final Integrated Solution

In [41]:
from datetime import datetime, timezone


class ApiClientConfig:
    DEFAULT_TIMEOUT = 30.0

    def __init__(self, base_url, api_token=None, timeout=DEFAULT_TIMEOUT):
        self.audit_log = []
        self.base_url = base_url
        self.timeout = timeout
        if api_token is not None:
            self.api_token = api_token

    @property
    def base_url(self):
        return self._base_url

    @base_url.setter
    def base_url(self, value):
        value = str(value).strip()
        if not value.startswith(("http://", "https://")):
            raise ValueError("base_url must start with http:// or https://")
        self._base_url = value.rstrip("/")

    @base_url.deleter
    def base_url(self):
        raise AttributeError("base_url is required and cannot be deleted")

    def _invalidate_headers(self):
        self.__dict__.pop("_headers_cache", None)

    @property
    def api_token(self):
        if "_api_token" not in self.__dict__:
            raise AttributeError("api_token is not configured")
        return self._api_token

    @api_token.setter
    def api_token(self, value):
        if not isinstance(value, str) or len(value) < 8:
            raise ValueError("api_token must be at least 8 characters")
        self._api_token = value
        self._invalidate_headers()

    @api_token.deleter
    def api_token(self):
        existed = "_api_token" in self.__dict__

        if existed:
            del self._api_token
            self._invalidate_headers()

        self.audit_log.append({
            "event": "api_token_deleted",
            "existed_before": existed,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })

    @property
    def timeout(self):
        return self._timeout

    @timeout.setter
    def timeout(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool) or value <= 0:
            raise ValueError("timeout must be a positive number")
        self._timeout = float(value)

    @timeout.deleter
    def timeout(self):
        self._timeout = self.DEFAULT_TIMEOUT

    @property
    def headers(self):
        if "_headers_cache" not in self.__dict__:
            headers = {"Accept": "application/json"}

            if "_api_token" in self.__dict__:
                headers["Authorization"] = f"Bearer {self._api_token}"

            self._headers_cache = headers

        # Return a copy so callers cannot mutate the cache accidentally.
        return dict(self._headers_cache)

    def reset_optional_state(self):
        # Public deletion semantics are intentionally reused.
        del self.api_token
        del self.timeout


config = ApiClientConfig(
    "https://api.example.com/",
    api_token="token-123456",
    timeout=10,
)

print("Base URL:", config.base_url)
print("Headers:", config.headers)
print("Timeout:", config.timeout)

del config.api_token

print("Headers after token deletion:", config.headers)

try:
    print(config.api_token)
except AttributeError as exc:
    print(exc)

del config.timeout
print("Timeout reset:", config.timeout)

try:
    del config.base_url
except AttributeError as exc:
    print(exc)

print("Audit log:")
for event in config.audit_log:
    print(event)

Base URL: https://api.example.com
Headers: {'Accept': 'application/json', 'Authorization': 'Bearer token-123456'}
Timeout: 10.0
Headers after token deletion: {'Accept': 'application/json'}
api_token is not configured
Timeout reset: 30.0
base_url is required and cannot be deleted
Audit log:
{'event': 'api_token_deleted', 'existed_before': True, 'timestamp': '2026-09-18T14:18:55.607306+00:00'}


# Extra Practice Problems

Try implementing these without looking back at the previous solutions.

### Practice A
Create a `BankProfile` whose `beneficiary` property can be deleted only when there are no pending transfers.

### Practice B
Create a `SearchQuery` whose `filters` property is cached after normalization. Deleting it should restore an empty filter set.

### Practice C
Create a `TemperatureReading` where deleting `celsius` also deletes cached Fahrenheit and Kelvin values.

### Practice D
Create a `CredentialStore` descriptor that automatically logs every deletion with the field name.

### Practice E
Create a subclass that narrows deletion permissions from its base class.

### Practice F
Create a property backed by a sentinel so that `None`, `0`, and `False` remain valid values distinct from "deleted".

### Practice G
Create a slotted class with two deletable properties and no `__dict__`.

### Practice H
Write tests proving that `delattr(obj, name)` and `del obj.name` produce identical side effects.

### Practice I
Create a delete-and-regenerate property for a random nonce or UUID.

### Practice J
Create an object where deleting one property is forbidden if another property currently depends on it.

# Summary

Property deletion is a powerful part of Python's descriptor model.

The syntax:

```python
del obj.some_property
```

should be understood as a **behavioral hook**, not merely as removal of a dictionary key.

A well-designed deleter can:

- remove backing state,
- reset a default,
- revoke optional data,
- enforce invariants,
- clear caches,
- close owned resources,
- archive old values,
- log events,
- or reject deletion entirely.

The most maintainable designs make those semantics explicit, predictable, and testable.